# Session 13 · Case study: error analysis of 20 misclassified reviews

**Block 3 practice.** Analyse 20 misclassified reviews, look at the most informative n-grams, and use what you find to improve the classifier.

Theory: [03-error-analysis-and-limits.md](../theory/03-error-analysis-and-limits.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
reviews = pd.read_parquet(DATA / "train_sample.parquet")
texts = reviews["title"] + " " + reviews["text"]          # one document = title + text
y = reviews["label"]
print(reviews.shape, y.value_counts(normalize=True).round(3).to_dict())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

idx_tr, idx_va = train_test_split(reviews.index, test_size=0.2, stratify=y, random_state=0)


def make_model(**tfidf_kw):
    kw = dict(ngram_range=(1, 2), min_df=3, sublinear_tf=True) | tfidf_kw
    return make_pipeline(TfidfVectorizer(**kw),
                         LogisticRegression(C=4, class_weight="balanced", max_iter=2000))


model = make_model().fit(texts[idx_tr], y[idx_tr])
pred = model.predict(texts[idx_va])
print("macro-F1:", round(f1_score(y[idx_va], pred, average="macro"), 3))
ConfusionMatrixDisplay.from_predictions(y[idx_va], pred, normalize="true", values_format=".2f");

## 1. Draw 20 errors and categorise them

The seed is fixed so that everyone in the class reads the same reviews.

In [ ]:
pd.set_option("display.max_colwidth", 400)
val = pd.DataFrame({"text": texts[idx_va], "rating": reviews.loc[idx_va, "rating"],
                    "true": y[idx_va], "pred": pred})
wrong = val[val["true"] != val["pred"]]
print(len(wrong), "errors")
errors20 = wrong.sample(20, random_state=1).copy()
errors20["category"] = ""          # fill in below
errors20[["rating", "true", "pred", "text"]]

Read each review and give it **one** category. Suggested categories (add your own):

- `mismatch`: text and rating disagree
- `mixed`: praise and complaint in one review
- `comparison`: compares with another product
- `negation`: negation, hedging or very short text
- `off-topic`: delivery, packaging, seller
- `noise`: the label is probably wrong
- `unclear`: no clear reason

In [ ]:
# fill in one category per review, in the order of the table above
my_categories = [""] * 20
errors20["category"] = my_categories
errors20["category"].value_counts()

## 2. What does the model rely on?

In [ ]:
vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
names = vec.get_feature_names_out()
top = {label: list(names[np.argsort(clf.coef_[k])[::-1][:15]]) for k, label in enumerate(clf.classes_)}
pd.DataFrame(top)

**Question.** Several of the top n-grams are titles such as "Three Stars". Count how often such titles occur in the training sample and in the test set, and by year.

In [ ]:
pattern = r"(?:one|two|three|four|five) stars?"
star = reviews["title"].str.fullmatch(pattern, case=False)
test = pd.read_parquet(DATA / "test.parquet")
print("sample:", int(star.sum()), f"({star.mean():.1%})",
      "| test:", int(test["title"].str.fullmatch(pattern, case=False).sum()))
pd.crosstab(pd.to_datetime(reviews["date"]).dt.year, star).tail(8)

## 3. Improve the classifier

Evaluate every change on the validation reviews **without** automatic star titles: these titles hardly occur in the test years, so a score that includes them is too optimistic.

In [ ]:
va_clean = [i for i in idx_va if not star[i]]
texts_blank = reviews["title"].where(~star, "") + " " + reviews["text"]   # star titles removed


def evaluate(name, docs, **tfidf_kw):
    m = make_model(**tfidf_kw).fit(docs[idx_tr], y[idx_tr])
    p_all = m.predict(docs[idx_va])
    p_clean = m.predict(docs[va_clean])
    return {"change": name,
            "F1 all validation": f1_score(y[idx_va], p_all, average="macro"),
            "F1 without star titles": f1_score(y[va_clean], p_clean, average="macro")}


results = [
    evaluate("baseline", texts),
    evaluate("star titles removed in training", texts_blank),
    evaluate("+ min_df=2", texts_blank, min_df=2),
]
pd.DataFrame(results).round(3)

## Your turn

1. Pick the largest *fixable* category from your 20 reviews and design one change for it (examples: character n-grams with `analyzer="char_wb"`, `ngram_range=(1, 3)`, a different `C`, `LinearSVC`, adding the title as a separate feature with `ColumnTransformer`). Add it to `results` above.
2. Is the improvement larger than the noise? Repeat the best two settings with three different `random_state` values for the split and compare the spread.
3. If a change helps on the clean validation reviews, refit it on all sample reviews and submit (see workbook 05).
4. Write three sentences for your team log: the most common error category, the change you tried, and its effect.

In [ ]:
# your code here